# 0 - Provision combined-experiment system node

Use this notebook in the Chameleon Jupyter environment to provision the system VM for the combined serving experiment.

Combined option used in this workflow:
- model-level: `smartqueue_ranker_dynamic_q.onnx`
- system-level: `UVICORN_WORKERS=4`
- result folder: `serving/evaluation/results/onnx_dynamic_int8_concurrent/`


In [ ]:
# runs in Chameleon Jupyter environment
from chi import server, context, lease, network
import chi, os, datetime

context.version = "1.0"
context.choose_project()
context.choose_site(default="KVM@TACC")

username = os.getenv("USER")
PROJECT_SUFFIX = "proj13"
LEASE_NAME = f"lease-serve-combined-{username}-{PROJECT_SUFFIX}"
SERVER_NAME = f"node-serve-combined-{username}-{PROJECT_SUFFIX}"
FLAVOR_NAME = "m1.medium"
IMAGE_NAME = "CC-Ubuntu24.04"


In [ ]:
# runs in Chameleon Jupyter environment
l = lease.Lease(LEASE_NAME, duration=datetime.timedelta(hours=8))
l.add_flavor_reservation(id=chi.server.get_flavor_id(FLAVOR_NAME), amount=1)
l.submit(idempotent=True)
l.show()


In [ ]:
# runs in Chameleon Jupyter environment
s = server.Server(
    SERVER_NAME,
    image_name=IMAGE_NAME,
    flavor_name=l.get_reserved_flavors()[0].name,
)
s.submit(idempotent=True)


In [ ]:
# runs in Chameleon Jupyter environment
s.associate_floating_ip()
s.refresh()
s.check_connectivity()


In [ ]:
# runs in Chameleon Jupyter environment
security_groups = [
    {"name": "allow-ssh", "port": 22, "description": "Enable SSH traffic on TCP port 22"},
    {"name": "allow-8888", "port": 8888, "description": "Enable TCP port 8888 (used by Jupyter)"},
    {"name": "allow-8000", "port": 8000, "description": "Enable TCP port 8000 (FastAPI)"},
]

for sg in security_groups:
    secgroup = network.SecurityGroup({
        "name": sg["name"],
        "description": sg["description"],
    })
    secgroup.add_rule(direction="ingress", protocol="tcp", port=sg["port"])
    secgroup.submit(idempotent=True)
    s.add_security_group(sg["name"])

print(f"updated security groups: {[sg['name'] for sg in security_groups]}")
s.refresh()
s.show(type="widget")


In [ ]:
# runs in Chameleon Jupyter environment
s.execute("curl -sSL https://get.docker.com/ | sudo sh")
s.execute("sudo groupadd -f docker; sudo usermod -aG docker $USER")
s.execute("sudo apt-get install -y docker-compose-plugin")


In [ ]:
# runs in Chameleon Jupyter environment
REPO_URL = "https://github.com/yanghao13111/SmartQueue.git"
s.execute(f"git clone {REPO_URL} ~/smartqueue")


Open an SSH session from your local terminal:

```bash
ssh -i ~/.ssh/id_rsa_chameleon cc@A.B.C.D
```

Replace `A.B.C.D` with the floating IP shown above.


From the VM host shell, start the combined FastAPI option:

```bash
cd ~/smartqueue/serving/docker
MODEL_PATH=/app/model_artifacts/smartqueue_ranker_dynamic_q.onnx MODEL_VERSION=onnx_dynamic_int8_concurrent UVICORN_WORKERS=4 docker compose -f docker-compose-system.yaml up -d --build --force-recreate fastapi jupyter
```

Health check:

```bash
curl http://localhost:8000/health
```

Run the scripted evaluation and save results under `onnx_dynamic_int8_concurrent`:

```bash
MODEL_PATH=/app/model_artifacts/smartqueue_ranker_dynamic_q.onnx MODEL_VERSION=onnx_dynamic_int8_concurrent UVICORN_WORKERS=4 docker compose --profile eval -f docker-compose-system.yaml run --rm eval sh run_evaluation.sh onnx_dynamic_int8_concurrent
```


After the stack is up, open Jupyter from the VM and run notebook:

- `7_combined_dynamic_int8_concurrent.ipynb`


In [ ]:
# runs in Chameleon Jupyter environment
# delete the VM when finished to keep costs low
s.delete()
